# 第2回：予測モデルを作成

この回は3つのパートで構成します：**何を、いつ、何のために予測するか ／ 数値を予測する—回帰 ／ 複数のモデルを同じ条件で比較する**。

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

**AIと一緒に進める**：分からないコードは、セル全体ではなく気になる数行をM365 CopilotなどのAIへ貼り、
説明や修正を相談します。ただし、提案されたコードは必ず実行結果を見て確かめます。

まず「基本」と「演習」を進めます。「補足」は必要に応じて読み、
「発展（任意）」「追加演習（任意）」「自由課題（任意）」は飛ばしても構いません。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回で扱うこと

何を・いつ予測するかを決めてデータリークを見抜き、回帰・分類のモデルを作り、様々なモデルの特徴を知ります。

### 進め方

この回は3つのパートに分かれています。パート1から順に「基本」と「演習」を進めてください。
1日で終える必要はありません。「発展（任意）」と「追加演習（任意）」は、余裕がある場合だけ取り組みます。

### 用語について

初めて出る用語は、その用語を使うセルで説明します。ここでまとめて暗記する必要はありません。

> **実行前の30秒予想**：各パートの問いに、今の言葉で仮の答えを書いてから始めます。


---

# パート1：何を、いつ、何のために予測するか

**このパートの問い：モデル構築より前に決めるべきことは何か。**


## 「良いモデル」の前に「正しい問い」

初心者がいちばん飛ばしがちで、実は最も効くのがこの回です。**どんなに精度が高くても、問いの立て方が
間違っていれば役に立ちません**。モデルを組む前に、次を1文で言えるようにします。

> **誰が・いつ・何を予測し・その結果をどう使うか。**

例：*実験条件を決める時点で使える情報から収率を予測し、優先して試す条件を選ぶ。*

ここで決定的に大事なのが**予測時点**です。「いつ予測するか」を決めると、その時点で**まだ手に入って
いない情報は使えない**と分かります。実験後にしか得られない値を入力に混ぜると、練習では高得点でも
本番でまったく使えない「ズル（リーク）」になります。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## 計画時に使える列／使えない列を仕分ける

このデータで「実験条件を決める時点」を予測時点とすると、収率・活性・純度・測定後シグナルは
**まだ存在しません**。使える列と使えない列を、はっきり2つのリストに分けます。この仕分けが
特徴量選びの土台になります。


In [ ]:
available_at_planning = [
    "scaffold_group", "solvent", "catalyst", "temperature_c", "reaction_time_h",
    "concentration_m", "molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds",
]
unavailable_at_planning = ["yield_pct", "active", "post_assay_signal", "purity_pct"]
print("計画時に使える列:", available_at_planning)
print("実験後に得られる列:", unavailable_at_planning)


### 読みどころ

`unavailable_at_planning`の列は「結果」や「結果に強く連動する測定値」です。これらを特徴量に入れると
リークになります。**列の名前ではなく「その値がいつ確定するか」で判断する**のがコツです。


## 演習：まず「単純な基準（ベースライン）」を作る

複雑なモデルに進む前に、**平均値だけ／多数派だけ**を答える最も単純なモデルを作ります。これが
比較の出発点（ものさし）になります。以降のどのモデルも、まずこれを超えることが最低条件です。


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.model_selection import train_test_split

train, valid = train_test_split(df, test_size=0.25, random_state=42)
reg = DummyRegressor(strategy="mean").fit(train[["molecular_weight"]], train["yield_pct"])
cls = DummyClassifier(strategy="most_frequent").fit(train[["molecular_weight"]], train["active"])
print("平均収率だけのMAE:", round(mean_absolute_error(valid["yield_pct"], reg.predict(valid[["molecular_weight"]])), 2))
print("多数派だけの正解率:", round(accuracy_score(valid["active"], cls.predict(valid[["molecular_weight"]])), 3))


### 出力の読み方

- **MAE（平均絶対誤差）**：予測が平均どれだけ外れるか。単位は収率と同じ%。「平均値だけ」でこの誤差、というものさしです。
- **多数派だけの正解率**が高く出ることに驚くかもしれません。活性が少ないデータでは「全部を多数派と答える」だけで正解率が高くなります。**だから正解率は当てにならない**。第3回パート2でF1を学ぶ動機になります。
- 本命モデルは、この2つの数字を**はっきり上回って初めて価値がある**と考えます。
- なお`Dummy`は答え(`y`)だけを見て予測するため、ここで渡している`molecular_weight`列の中身は使いません（形式的な引数です）。


## 補足：リーク候補を自動で洗い出す

「どの列がリークか」を人手で全部見るのは大変です。目的変数と**極端に強く連動する列**は、結果由来の
情報が紛れている疑いがあります。それを見つける監査を関数にしておくと、自社データでも使い回せます。


In [ ]:
def leakage_audit(frame, target: str, threshold: float = 0.9):
    "目的変数と相関が極端に高い数値列を、リーク候補として洗い出す。"
    numeric = frame.select_dtypes(include="number")
    corr = numeric.corrwith(frame[target]).abs().drop(labels=[target], errors="ignore")
    report = corr.sort_values(ascending=False).to_frame("|相関|")
    report["リーク候補"] = report["|相関|"] >= threshold
    return report.round(3)

display(leakage_audit(df, target="active", threshold=0.6))
print("post_assay_signalは測定後の値。相関が高くても計画時には使えない。")


### 出力の読み方と注意

- `active`との|相関|が高い順に並びます。`post_assay_signal`が上位に来るはず。これは**活性測定後の値**なので、計画時には存在せず、使えばリークです。
- ただしこの監査は**あくまで補助**。相関が低くてもリークする列（例：実験日から結果を推測できる場合）もあります。最終判断は「その値がいつ確定するか」で人が行います。
- `threshold`はリーク候補とみなす相関の閾値。厳しく見たいなら下げます。


## 演習：自分のテーマを1枚に整理する

次の8点を、機密を書かずに埋めます。**利用者／判断／予測時点／目的変数／使える列／使えない列／
回帰か分類か／単純な基準**。埋まらない項目があれば、それが今いちばん詰めるべき点です。

## Copilotへの相談

Copilotには、曖昧な項目を勝手に埋めさせず「確認すべき質問」の形で返すよう頼みます。


## 発展（任意）：最適な閾値は「コスト」で決まる

分類モデルは確率を出し、ある**閾値**を超えたら「活性」と判定します。既定の0.5が最適とは限りません。
最適な閾値は指標ではなく、**誤りのコスト**で決まります。ここでは「見逃し（偽陰性）＝有望条件を逃す損失」が
「偽陽性＝無駄な追試」より10倍高い状況を想定し、期待コストが最小になる閾値を探します。


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

feat = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_tr, X_te, y_tr, y_te = train_test_split(df[feat], df["active"], test_size=0.3, random_state=42, stratify=df["active"])
clf = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

cost_fn, cost_fp = 10, 1  # 見逃し=有望条件を逃す損失、偽陽性=無駄な追試
rows = []
for t in np.linspace(0.1, 0.9, 17):
    pred = (proba >= t).astype(int)
    fp = int(((pred == 1) & (y_te == 0)).sum())
    fn = int(((pred == 0) & (y_te == 1)).sum())
    rows.append({"閾値": round(t, 2), "偽陽性": fp, "偽陰性": fn, "期待コスト": fp * cost_fp + fn * cost_fn})
cost_table = pd.DataFrame(rows)
best = cost_table.loc[cost_table["期待コスト"].idxmin(), "閾値"]
display(cost_table)
print("コスト最小の閾値:", best, " / 見逃しが高いほど閾値は下がる")


### 出力の読み方

- 閾値を下げると「活性」と判定する数が増え、**偽陰性（見逃し）は減るが偽陽性は増える**トレードオフが表で見えます。
- 見逃しのコストが高いので、**最適閾値は0.5より低め**に出るはずです。「とりあえず0.5」がいかに恣意的かが分かります。
- コストの比（10:1）を変えれば最適閾値も動きます。**閾値はモデルの外側で、目的に合わせて選ぶ**ものだと理解できます。


### まとめ：指標は意思決定から逆算する

見逃しを避けたい探索段階なら**recall**寄り、追試コストが高い絞り込み段階なら**precision**寄り。
「良いスコア」を追うのではなく、「この予測で何を決め、間違えると何を失うか」から指標と閾値を選びます。
指標や閾値を選ぶときは、常にこの「何を決め、何を失うか」に立ち返ります。


## 追加演習（任意）

問題設定まわりのコードをもう少し。90分の外の自習向けです。まず**単純基準（Dummy）を戦略ごとに
比較**し、「どのベースラインを土俵にするか」を意識します。


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.model_selection import train_test_split

tr, va = train_test_split(df, test_size=0.25, random_state=42)
print("=== 回帰の単純基準 ===")
for strat in ["mean", "median"]:
    d = DummyRegressor(strategy=strat).fit(tr[["molecular_weight"]], tr["yield_pct"])
    print(f"{strat:14s} MAE={mean_absolute_error(va['yield_pct'], d.predict(va[['molecular_weight']])):.2f}")
print("=== 分類の単純基準 ===")
for strat in ["most_frequent", "stratified", "uniform"]:
    d = DummyClassifier(strategy=strat, random_state=42).fit(tr[["molecular_weight"]], tr["active"])
    print(f"{strat:14s} accuracy={accuracy_score(va['active'], d.predict(va[['molecular_weight']])):.3f}")


### 出力の読み方

- 回帰は`mean`と`median`でMAEが少し違います。分布が歪んでいると`median`が有利なことも。
- 分類の`most_frequent`は正解率が高く見えますが、これは第3回パート2で学ぶ「不均衡の罠」。`stratified`/`uniform`はランダムに近い基準です。
- 本命モデルは、**これらのうち最も手強い基準**を超えて初めて価値があります。


### 予測時点チェックを関数にする

第2回パート2で確認した「使える列／使えない列」を、候補リストから自動で仕分ける関数にします。自社データでも
使い回せる、実務的な安全装置です。


In [ ]:
def check_feature_timing(candidate_features, available_now, target):
    "特徴量候補を『使える/使えない(リーク)』に仕分ける。"
    rows = []
    for col in candidate_features:
        if col == target:
            verdict = "目的変数(使わない)"
        elif col in available_now:
            verdict = "使える"
        else:
            verdict = "使えない(予測時点で未確定)"
        rows.append({"列": col, "判定": verdict})
    return pd.DataFrame(rows)

check_feature_timing(
    ["temperature_c", "logp", "yield_pct", "post_assay_signal", "active"],
    available_now=available_at_planning,
    target="active",
)


### 出力の読み方

`yield_pct`や`post_assay_signal`が「使えない(予測時点で未確定)」と仕分けられます。列名を眺めるだけでなく、
**このチェックを通してから特徴量を確定する**運用にすれば、リークの多くを機械的に防げます。


### 期待値で「試すか否か」を決める

活性確率を予測できたとして、「その条件を追試すべきか」を**期待利益**で判断する簡単な例です。
確率×利益からコストを引いて、プラスなら試す。第2回パート2・第3回パート2のコスト最適閾値の考え方の土台です。


In [ ]:
import numpy as np

proba = np.array([0.10, 0.40, 0.60, 0.85])   # 各条件の活性確率（仮）
gain_if_active, cost_of_test = 100, 20
table = pd.DataFrame({"活性確率": proba})
table["期待利益"] = proba * gain_if_active - cost_of_test
table["試す?"] = table["期待利益"] > 0
table.round(1)


### 出力の読み方

期待利益がプラスの条件だけ「試す?=True」になります。ここでは損益分岐の確率は`cost/gain=0.2`。
つまり**活性確率20%以上なら試す**が最適で、これがそのまま判定閾値になります。「閾値0.5」が絶対でない
理由が、利益の式から自然に出てくることを確認してください。


---

# パート2：数値を予測する—回帰

**このパートの問い：連続値の予測モデルを、何と比べ、不確かさをどう示すか。**


## 回帰＝「数値そのもの」を予測する

ここまでは活性の有無（0/1）でしたが、この回は**収率（%）という連続値**を予測します。これを
**回帰**と呼びます。回帰でいちばん大事な問いは「その予測は**何と比べて**良いのか」。だから今回も
**平均値だけを返すベースライン**を必ず土俵に上げます。

まず下準備。日本語フォント設定と、使うモデルの読み込み、学習/検証の分割をまとめて行います。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
# グラフの日本語が文字化けしないようにする設定です。中身は今は理解しなくてOK、そのまま実行してください。
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["yield_pct"], test_size=0.25, random_state=42)


## 演習：4モデルを交差検証で、3つの指標で比べる

回帰の代表的な指標を先に押さえます。

- **MAE（平均絶対誤差）**：平均で何%外すか。単位が収率と同じで**いちばん直感的**。
- **RMSE**：大きな外れを二乗で重く見る。「たまの大外し」を嫌う場面向け。
- **R²（決定係数）**：平均値予測と比べてどれだけ説明できたか（1に近いほど良い、0は平均値並み）。

`neg_...`はsklearnの都合で「大きいほど良い」に符号反転された指標名。表示時に`-`で元へ戻します。
（scikit-learnの`scoring`は「大きいほど良い」に統一されているため、誤差系の指標は符号が反転しています。）

コード中の`make_pipeline(SimpleImputer(...), モデル)`は、**欠損補完とモデルを1つにまとめて「1個のモデル」の
ように扱う**ための道具です。こうすると`fit`/`predict`や交差検証がまとめて安全に回せます。仕組みは第3回パート3で
詳しく学ぶので、ここでは「前処理とモデルをセットにする書き方」とだけ捉えて大丈夫です。


In [ ]:
from sklearn.model_selection import cross_validate, KFold

models = {
    "平均値": DummyRegressor(),
    "線形回帰": make_pipeline(SimpleImputer(strategy="median"), LinearRegression()),
    "決定木": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=4, random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)),
}
cv = KFold(5, shuffle=True, random_state=42)
scoring = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}
rows = []
for name, estimator in models.items():
    res = cross_validate(estimator, df[features], df["yield_pct"], cv=cv, scoring=scoring)
    rows.append({"モデル": name, "MAE": -res["test_MAE"].mean(), "RMSE": -res["test_RMSE"].mean(), "R2": res["test_R2"].mean()})
pd.DataFrame(rows).sort_values("MAE").round(3)


### 出力の読み方

- MAEの小さい順に並びます。**「平均値」より各モデルがどれだけMAEを下げたか**が価値です。下げ幅が小さいなら、その特徴量では収率を説明しきれていません。
- MAEとRMSEの差が大きいモデルは、**たまに大きく外している**サイン（RMSEが大外れを強調するため）。
- R²が0近くなら「平均値と大差ない」、負なら「平均値より悪い」。**まずベースライン超え**を確認します。
- なお、この`cross_validate`は内部でデータを分割し直して評価します。上のセルで作った`X_train`/`X_valid`はここでは使わず、この後の**残差図**（予測と実測を見る図）で使います。


## 予測と実測、そして「残差」を絵で見る

数字だけでなく図で確かめます。左は**予測と実測の散布図**（点が対角線に乗るほど良い）、右は
**残差図**（実測−予測を予測値に対してプロット）。残差は0の周りに**模様なくばらける**のが理想です。
偏りや傾きがあれば、モデルが取りこぼした構造があります。


In [ ]:
rf = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)).fit(X_train, y_train)
pred = rf.predict(X_valid)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_valid, pred, alpha=0.6)
axes[0].plot([y_valid.min(), y_valid.max()], [y_valid.min(), y_valid.max()], "--")
axes[0].set(xlabel="実測収率", ylabel="予測収率", title="予測と実測")
axes[1].scatter(pred, y_valid - pred, alpha=0.6)
axes[1].axhline(0, linestyle="--")
axes[1].set(xlabel="予測収率", ylabel="残差（実測-予測）", title="残差")
plt.tight_layout()


### 出力の読み方

- **左**：点が破線（予測＝実測）に近いほど良い。高収率の領域で点が下に外れていれば、**高い収率を低めに予測しがち**という癖です。
- **右**：残差が予測値によって偏る（例：予測が大きいほど残差が下がる）なら、モデルが端の領域を苦手にしています。**きれいな水平の帯**が理想。
- 図は「どこで外すか」を教えてくれます。次のセルで、実際に大きく外した試料を取り出します。


## 大きく外した試料を名指しで調べる

平均のMAEでは見えない「個別の大外し」を確認します。絶対誤差の大きい上位8件を、系列や触媒つきで
取り出すと、**特定の条件で外していないか**の手がかりになります。


In [ ]:
errors = df.loc[y_valid.index, ["sample_id", "scaffold_group", "catalyst", "yield_pct"]].copy()
errors["予測"] = pred
errors["絶対誤差"] = (errors["yield_pct"] - errors["予測"]).abs()
errors.nlargest(8, "絶対誤差").round(2)


### 出力の読み方

大外し8件に**同じ系列や同じ触媒が偏っていないか**を見ます。偏っていれば、その条件を表す特徴量が
足りない可能性（第4回パート2の特徴量設計の動機）。ばらばらなら、単なるノイズかもしれません。

## 変更して確認

`max_depth=6`を`3`や`10`へ変え、MAEの表・残差図・大外し試料が**どう連動して動くか**を観察します。


## 発展（任意）：伸び悩みの原因と、予測の不確かさ

発展として3つ。**学習曲線**（データを増やせば改善するか）、**群別残差**（どの系列で系統的に外すか）、
**予測区間**（1点の予測に幅を添える）です。


### 学習曲線：データ不足か、表現力不足か

「もっとデータを集めれば精度が上がる？」に答える図です。学習データ量を増やしながら、学習MAEと
検証MAEの推移を描きます。2本が近づいて高止まりなら**データ追加は効きにくい**（特徴量やモデルを
見直すべき）。2本が離れて検証MAEがまだ下がりそうなら**データ追加が効く**サインです。


In [ ]:
import numpy as np
from sklearn.model_selection import learning_curve

sizes, train_scores, valid_scores = learning_curve(
    make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)),
    df[features], df["yield_pct"], cv=5, scoring="neg_mean_absolute_error",
    train_sizes=np.linspace(0.2, 1.0, 5),
)
plt.figure(figsize=(7, 4))
plt.plot(sizes, -train_scores.mean(1), "o-", label="学習MAE")
plt.plot(sizes, -valid_scores.mean(1), "o-", label="検証MAE")
plt.xlabel("学習データ件数"); plt.ylabel("MAE"); plt.legend(); plt.title("学習曲線")
plt.tight_layout()
print("2本が高止まりで近いなら、データ追加より特徴量やモデルを見直す。")


### 出力の読み方

- 右端（全データ使用）で**検証MAEがまだ下降中**なら、データを増やす価値あり。**水平に寝ている**なら頭打ち。
- 学習MAEと検証MAEの**縦の隙間**が過学習の程度。隙間が大きいほど「覚えすぎ」寄りです。


### 群別に残差を見る：どの系列で偏るか

全体のMAEが良くても、特定の化合物系列だけ系統的に外していることがあります。系列ごとに件数・MAE・
**平均残差**（符号つき）を出すと、「この系列を平均的に低く見積もっている」といった偏りが見えます。


In [ ]:
errors["残差"] = errors["yield_pct"] - errors["予測"]
group_error = errors.groupby("scaffold_group").agg(件数=("残差", "size"), MAE=("絶対誤差", "mean"), 平均残差=("残差", "mean"))
display(group_error.sort_values("MAE", ascending=False).round(2))
print("平均残差が正なら、その系列を平均的に過小予測している。")


### 出力の読み方

**平均残差が0から大きく離れた系列**が要注意。正なら過小予測、負なら過大予測です。件数が少ない系列は
偶然も大きいので、件数と併せて読みます。系統的な偏りは、その系列を表す特徴量の不足を示唆します。


### 予測区間：1点でなく「幅」で答える

「収率は62%」より「10〜90%の確率で50〜74%」の方が、意思決定に誠実なことがあります。**分位点回帰**で
下限（10%点）と上限（90%点）を別々に予測し、実測が本当にその区間に入る割合（**被覆率**）を検証します。


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

low = HistGradientBoostingRegressor(loss="quantile", quantile=0.1, max_iter=200, random_state=42).fit(X_train, y_train)
high = HistGradientBoostingRegressor(loss="quantile", quantile=0.9, max_iter=200, random_state=42).fit(X_train, y_train)
low_pred, high_pred = low.predict(X_valid), high.predict(X_valid)
coverage = ((y_valid.to_numpy() >= low_pred) & (y_valid.to_numpy() <= high_pred)).mean()
print(f"10-90%予測区間の実測被覆率: {coverage:.1%}（理想は約80%）")
pd.DataFrame({"実測": y_valid.to_numpy()[:8], "下限": low_pred[:8].round(1), "上限": high_pred[:8].round(1)})


### 出力の読み方

- 10〜90%区間なので、被覆率は**理想80%**に近いほど区間が正直。大きく下回れば区間が狭すぎ（自信過剰）、上回れば広すぎです。
- 表の8件で、**実測が下限〜上限に収まっているか**を目で確認します。幅の広い試料はモデルが自信を持てていない試料です。


## 追加演習（任意）

線形モデルの正則化や非線形化を試します。90分の外の自習向けです。まず**RidgeとLasso**を、正則化の
強さ`alpha`を変えて比較します（`alpha`が大きいほど係数を抑え、過学習を防ぐ）。


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(5, shuffle=True, random_state=42)
rows = []
for alpha in [0.01, 0.1, 1.0, 10.0]:
    for name, reg in {"Ridge": Ridge(alpha=alpha), "Lasso": Lasso(alpha=alpha, max_iter=5000)}.items():
        pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), reg)
        mae = -cross_val_score(pipe, df[features], df["yield_pct"], cv=cv, scoring="neg_mean_absolute_error").mean()
        rows.append({"モデル": name, "alpha": alpha, "MAE": mae})
pd.DataFrame(rows).pivot(index="alpha", columns="モデル", values="MAE").round(3)


### 出力の読み方

`alpha`を変えるとMAEが変わり、最適な強さがあることが分かります。強すぎると単純になりすぎ（未学習）、
弱すぎると過学習寄り。RidgeとLassoで最適`alpha`が違うのも普通です。**正則化は複雑さを調整するダイヤル**です。


### 多項式特徴量で「曲がり」を線形モデルに教える

線形回帰は直線しか引けませんが、`PolynomialFeatures`で二乗や交互作用の列を足すと、曲がった関係も
表せます。次数を上げすぎると過学習するので、交差検証で確かめます。


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

for degree in [1, 2, 3]:
    pipe = make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(),
        PolynomialFeatures(degree, include_bias=False), LinearRegression(),
    )
    mae = -cross_val_score(pipe, df[features], df["yield_pct"], cv=cv, scoring="neg_mean_absolute_error").mean()
    print(f"多項式次数{degree}: MAE={mae:.3f}")


### 出力の読み方

次数2で、温度の山型（第2回パート1）を線形モデルが表せるようになり、MAEが下がることが多いはず。ただし次数3で
悪化したら過学習のサイン。「複雑にすれば良い」ではなく、**交差検証が下がる範囲でだけ複雑にする**が原則です。


### 部分依存プロット：モデルは各変数をどう使っているか

`PartialDependenceDisplay`は、「他を平均的に保ったまま、ある変数を動かすと予測がどう変わるか」を
描きます。モデルが温度の山型を学べているかを、目で確認できます。


In [ ]:
from sklearn.inspection import PartialDependenceDisplay

rf = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)).fit(df[features], df["yield_pct"])
PartialDependenceDisplay.from_estimator(rf, df[features], ["temperature_c", "concentration_m"])
plt.tight_layout()


### 出力の読み方

温度の曲線が**山型**（中ほどで予測収率が最大）になっていれば、モデルは第2回パート1で見た構造を学べています。
部分依存プロットは、ブラックボックスに見える木モデルの「考え方」を説明する強力な道具で、
研究者への説明資料としても有効です。


---

# パート3：複数のモデルを同じ条件で比較する

**このパートの問い：複雑なモデルは本当にいつも優れているか。**


## 「複雑なモデルほど強い」は本当か

新しいモデルを次々試したくなりますが、この回で確かめるのは**「複雑さは必ずしも勝たない」**という
実感です。大事なのは勝ち負けそのものより、**フェアな比べ方**を身につけること。フェアな比較には
3つの「同じ」が要ります：**同じ分割・同じ指標・同じ前処理**。

まず、単純〜複雑まで5つのモデルを1つの辞書にまとめます。前処理が要るモデルは`make_pipeline`で
前処理込みにしてあるので、どれも同じ`X_train`をそのまま渡せます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import time
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["active"], test_size=0.25, random_state=42, stratify=df["active"])
models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)),
    "Tree": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=4, random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=200, random_state=42),
}


## 演習：同じ土俵で、F1と学習時間を並べる

5モデルを同じデータで学習し、**検証F1**と**学習にかかった秒数**を並べます。性能だけでなく
**コスト（時間）**も一緒に見るのが実務的な比較です。


In [ ]:
rows = []
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    rows.append({"モデル": name, "検証F1": f1_score(y_valid, model.predict(X_valid)), "学習秒": elapsed})
pd.DataFrame(rows).sort_values("検証F1", ascending=False).round({"検証F1": 3, "学習秒": 4})


### 出力の読み方

- **Dummyが最下位**なのは当然。他がDummyをどれだけ引き離すかが価値です。
- **最も複雑なモデルが1位とは限りません**。線形モデルが健闘したり、木モデルと僅差だったりします。差が小さいなら、**速くて説明しやすいモデル**を選ぶ理由になります。
- ただし、これは**1回の分割の結果**。順位が分割運で入れ替わるかもしれません。次で交差検証により安定性を確かめます。


## 補足：交差検証で「安定して強いか」を見る

1回の勝敗は運に左右されます。交差検証で**平均F1・ばらつき(標準偏差)・最低F1**を出し、
「平均が高い」だけでなく「**転んでも大崩れしない**（最低F1が高い）」モデルを評価します。


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(5, shuffle=True, random_state=42)
stability = []
for name, estimator in models.items():
    scores = cross_val_score(estimator, df[features], df["active"], cv=cv, scoring="f1")
    stability.append({"モデル": name, "F1平均": scores.mean(), "F1標準偏差": scores.std(), "最低F1": scores.min()})
pd.DataFrame(stability).sort_values("F1平均", ascending=False).round(3)


### 出力の読み方

- **F1平均**で総合力、**F1標準偏差**で安定度、**最低F1**で最悪ケースを見ます。
- 平均が僅差なら、**標準偏差が小さい方**が実務では安心。平均1位でも最低F1が極端に低いモデルは、条件次第で大外しする危険があります。
- 1回の分割（前セル）と順位が入れ替わることもあります。だから**単発の勝敗で決めない**。これがこの回の教訓です。


## 5人の担当

Dummy / Logistic / Tree / Random Forest / Gradient Boosting を1人ずつ担当し、
**スコア・学習時間・説明しやすさ・安定性**を1行で共有します。「どれが最強か」ではなく
「**この用途にはどれが妥当か**」を言葉にすることが到達目標です。


## 発展（任意）：性能差とモデルを組み合わせる効果を確認する

上位2モデルのF1差が0.01だったとして、それは本物の差でしょうか、それとも分割運でしょうか。
ここでは**反復交差検証＋統計的検定**で差の確からしさを測り、次に複数モデルを組み合わせたときの効果を確認します。


### 反復CV＋Wilcoxon検定：差は偶然でないか

分割の乱数を変えて交差検証を何度も繰り返し（反復CV）、上位2モデルのスコア列を**対応のある検定
（Wilcoxon）**で比べます。p値が小さいほど「差は偶然では説明しにくい」と読めます。


In [ ]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from scipy.stats import wilcoxon

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=42)
dist = {name: cross_val_score(est, df[features], df["active"], cv=rcv, scoring="f1") for name, est in models.items()}
summary = pd.DataFrame({name: {"F1平均": s.mean(), "F1_SD": s.std()} for name, s in dist.items()}).T.sort_values("F1平均", ascending=False)
display(summary.round(3))
top2 = summary.index[:2].tolist()
stat, p = wilcoxon(dist[top2[0]], dist[top2[1]])
print(f"{top2[0]} vs {top2[1]} のWilcoxon検定 p={p:.3f}（小さいほど差が偶然でない）")


### 出力の読み方

- **p値が0.05より大きい**なら、上位2モデルの差は「偶然の範囲」かもしれず、**わざわざ複雑な方を選ぶ理由は弱い**。
- p値が小さくても、差の**大きさ**（実務的な意味があるか）は別問題。「統計的に有意」と「実務的に重要」は違う、という感覚を持ちます。
- 補足：反復交差検証のスコアは同じデータを使い回すため完全には独立でなく、素朴な検定のp値は**楽観的（有意に出やすい）**になりがちです。ここでは大まかな目安として読み、断定の根拠には使いません。


### 投票・スタッキングで組み合わせる

間違え方の違うモデルを組み合わせると、単体より安定することがあります。**Voting**は予測確率の平均、
**Stacking**は各モデルの予測を入力に上位モデルで統合します。単体最良と比べます。


In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

estimators = [(name, est) for name, est in models.items() if name != "Dummy"]
ensembles = {
    "Voting(soft)": VotingClassifier(estimators, voting="soft"),
    "Stacking": StackingClassifier(estimators, final_estimator=LogisticRegression(max_iter=1000), cv=5),
}
for name, est in ensembles.items():
    scores = cross_val_score(est, df[features], df["active"], cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1")
    print(f"{name:14s} F1平均={scores.mean():.3f} ± {scores.std():.3f}")
print("最良単体:", summary.index[0], "F1平均=", round(summary.iloc[0, 0], 3))


### 出力の読み方

組み合わせたモデルが最良単体を**明確に上回るとは限りません**。効果が出やすいのは、元のモデルたちが
「互いに違う間違え方」をするとき。差がわずかなら、運用の手間を考えて単体を選ぶのも正解です。


### 任意：勾配ブースティング専用ライブラリ

XGBoostが入っていれば試します（`uv sync --extra advanced`）。無い環境では自動でメッセージを出して
スキップし、sklearnの`HistGradientBoosting`で代用できます。


In [ ]:
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42, eval_metric="logloss")
    scores = cross_val_score(xgb, df[features].fillna(df[features].median()), df["active"], cv=5, scoring="f1")
    print("XGBoost F1平均:", round(scores.mean(), 3))
except ImportError:
    print("XGBoostは任意です（uv sync --extra advanced）。HistGradientBoostingで代用できます。")


### 出力の読み方

XGBoostのF1が、既に見たGradient Boostingと**近い値**になるはずです。「専用ライブラリ＝必ず勝つ」では
ありません。ライブラリの新しさより、**フェアな比較の枠組み**の方がずっと大事だと、あらためて分かります。


## 追加演習（任意）

モデル比較をさらに多面的に。90分の外の自習向けです。まず2モデルの**学習曲線**を並べ、
「データ追加が効くタイプか」を比べます。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, ["Logistic", "Random Forest"]):
    sizes, tr, va = learning_curve(models[name], df[features], df["active"], cv=5, scoring="f1", train_sizes=np.linspace(0.2, 1.0, 5))
    ax.plot(sizes, tr.mean(1), "o-", label="学習")
    ax.plot(sizes, va.mean(1), "o-", label="検証")
    ax.set_title(name); ax.set_xlabel("学習件数"); ax.set_ylabel("F1"); ax.legend()
plt.tight_layout()


### 出力の読み方

- 線形モデル（Logistic）は学習と検証の差が小さい（過学習しにくい）が頭打ちも早い傾向。
- Random Forestは差が大きい（表現力が高く過学習寄り）が、データを増やすと伸びる余地があることも。
- **モデルによってデータ追加の効き方が違う**。増やすか、特徴量を工夫するかの判断材料になります。


### 複数指標を一度に比べる

F1だけでなく、precision・recall・ROC-AUCも同時に交差検証で出します。用途によって重視する指標が
違う（第3回パート2）ので、多面的に見て選びます。


In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold

cv = StratifiedKFold(5, shuffle=True, random_state=42)
scoring = ["f1", "precision", "recall", "roc_auc"]
rows = []
for name, est in models.items():
    if name == "Dummy":
        continue
    res = cross_validate(est, df[features], df["active"], cv=cv, scoring=scoring)
    rows.append({"モデル": name, **{m: res[f"test_{m}"].mean() for m in scoring}})
pd.DataFrame(rows).round(3)


### 出力の読み方

あるモデルはrecallが高くprecisionが低い、別のモデルは逆、ということが起きます。**単一のF1では隠れる
個性**が見えます。「見逃しを避けたい」ならrecall列、「空振りを避けたい」ならprecision列で選びます。


### 性能とコストの釣り合い（木の本数）

木の本数（`n_estimators`）を増やすと精度は上がりやすい一方、学習時間も延びます。どこで頭打ちになるかを
見て、**費用対効果**で選びます。


In [ ]:
import time
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score

rows = []
for n in [50, 100, 200, 400]:
    est = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=n, max_depth=6, random_state=42))
    t = time.perf_counter(); est.fit(X_train, y_train); sec = time.perf_counter() - t
    rows.append({"n_estimators": n, "学習秒": sec, "検証F1": f1_score(y_valid, est.predict(X_valid))})
pd.DataFrame(rows).round({"学習秒": 4, "検証F1": 3})


### 出力の読み方

F1はある本数で頭打ちになり、その先は時間だけ延びるはず。**「もう増やしても得しない」点**を見つけるのが
チューニングの勘所。本番のデータ量ではこの差が大きくなるので、小さいうちに感覚をつかんでおきます。


---

## よくある誤り

- 入手できる列をすべて使う
- 目的変数が測定や運用で不安定
- 精度目標だけで利用方法とコストが決まっていない
- R²だけで利用可能と判断する
- テストデータでモデルを選ぶ
- 点予測だけを示し不確かさを伝えない
- 異なる分割で比較する
- モデルごとに異なる指標を報告する
- 最も高い1回のスコアだけを採用する

## 自習（任意・30〜60分）

- 自社テーマを機密情報なしで問題設定キャンバスへ落とす
- 偽陽性・偽陰性のコストを入れ、期待コスト最小の閾値を計算する
- 学習曲線を描き、データ追加が効くかを1文で判断する
- 分位点回帰の10-90%区間の被覆率を検証データで確認する
- RepeatedStratifiedKFoldでF1分布を作り、上位2モデルをWilcoxon検定で比べる
- StackingClassifierと最良単体のF1・学習時間を比較する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 誰が何を判断するモデルか
2. 予測時点で本当に得られる列はどれか
3. コスト行列から最適な閾値をどう求めるか
4. MAEとRMSEは何を違って重視するか
5. 学習曲線から何を読み取れるか
6. 予測区間が点予測より役立つ場面はどこか
7. 公平な比較に固定すべきものは何か
8. 対応のある検定が必要な理由は何か
9. スタッキングが効きやすいのはどんなときか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
